# LAB | Week 2 | Day 1 | API Calling to ChatGPT
## John Adams

**Description:**
Build an automation system that takes a product image and basic metadata. Process the image in ChatGPT with a prompt to output a professionally written product listing and formats it in an SEO friendly way for use in your e-commerce platform.


## Step 1: Setting Up the Project

In [ ]:
# Install the required packages (commenting out the ones already done)
# !pip install openai pydantic python-dotenv
# !pip install datasets

In [3]:
# Import ChatGPT libraries
import os
from openai import OpenAI
from pydantic import BaseModel
from typing import List
import json

# Initialize client
from dotenv import load_dotenv

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))



## Step 2: Preparing the Dataset

In [4]:
# Install: pip install datasets
from datasets import load_dataset
import requests
from PIL import Image
import pandas as pd
from pathlib import Path

# Load dataset from HuggingFace
print("Loading product dataset...")
try:
    dataset = load_dataset("cvnberk/amazon-products", split="train[:5]")
    print(f"✓ Loaded {len(dataset)} products")

    products_df = pd.DataFrame(dataset)
    print(f"Dataset columns: {products_df.columns.tolist()}")

except Exception as e:
    print(f"⚠ Could not load HuggingFace dataset: {e}")
    print("Using local images instead...")

    # Alternative: Use local images
    products_data = [
        {
            "id": 1,
            "name": "Wireless Headphones",
            "price": 79.99,
            "category": "Electronics",
            "image_path": "images/product1.jpg"
        },
        # Add more products...
    ]

    products_df = pd.DataFrame(products_data)

# Create images directory
images_dir = Path("product_images")
images_dir.mkdir(exist_ok=True)

print(f"\n✓ Dataset prepared!")
print(f"  Total products: {len(products_df)}")

# Visualizing the first few products in a table
products_df.head()

Loading product dataset...


✓ Loaded 5 products
Dataset columns: ['Product Name', 'Category', 'Description', 'Selling Price', 'Product Specification', 'Image']

✓ Dataset prepared!
  Total products: 5


,Product Name,Category,Description,Selling Price,Product Specification,Image
0,Craft-tastic – Empower Poster – Craft Kit – De...,Toys & Games | Arts & Crafts | Craft Kits | Pa...,PERFECT GIFT FOR AGES 8 AND ABOVE: Make this f...,$14.47,ProductDimensions:3x10x15inches|ItemWeight:15....,https://images-na.ssl-images-amazon.com/images...
1,Melissa & Doug Dot-to-Dot# & Letter Coloring P...,Toys & Games | Games & Accessories | Board Games,3 jumbo connect-the-dots coloring pads ABC far...,$12.74,ProductDimensions:11x0.8x14inches|ItemWeight:3...,https://images-na.ssl-images-amazon.com/images...
2,"RPM Rear Shock Tower for The Nitro Slash, Nitr...",Toys & Games | Hobbies | Remote & App Controll...,Great Condition.,$9.06,ProductDimensions:5.9x4x0.4inches|ItemWeight:0...,https://images-na.ssl-images-amazon.com/images...
3,Disney Pixar Cars Mini Racers Crank & Crash De...,Toys & Games | Play Vehicles | Toy Vehicles,Disney/Pixar Cars 3 new Crazy 8 track.,$27.85,ProductDimensions:2.9x14x10inches|ItemWeight:1...,https://images-na.ssl-images-amazon.com/images...
4,Areaware Cubebot Small,Toys & Games | Puzzles | Brain Teasers | Assem...,Great Condition.,$28.92,NaN,https://images-na.ssl-images-amazon.com/images...


**Observations:**
- Per teaching instructions, I'm using 5 sample and will scale up from there after verifying the model is working and output is as expected

## Step 3: Encoding Images for API

In [5]:
# Download the image from its URL, then encode it to base64
import io, base64
import requests
from PIL import Image

def encode_image_to_base64(image_url, min_size=512):
    """Download an image from a URL and encode it to a base64 string.
    Upscales small images before encoding, since OpenAI vision struggles
    with tiny images."""
    response = requests.get(image_url, timeout=10)
    image = Image.open(io.BytesIO(response.content)).convert("RGB")

    if min(image.size) < min_size:
        scale = min_size / min(image.size)
        new_size = (int(image.width * scale), int(image.height * scale))
        image = image.resize(new_size, Image.LANCZOS)

    buffer = io.BytesIO()
    image.save(buffer, format="JPEG")
    img_str = base64.b64encode(buffer.getvalue()).decode("utf-8")
    return img_str

# Example usage
sample_image_url = products_df.iloc[0]["Image"]
encoded_image = encode_image_to_base64(sample_image_url)
print(f"Encoded image length: {len(encoded_image)} characters")
print(f"Encoded prefix: {encoded_image[:40]}...")

# Free local verification: decode it back and confirm it's a valid image
decoded = base64.b64decode(encoded_image)
test_img = Image.open(io.BytesIO(decoded))
test_img.verify()
test_img = Image.open(io.BytesIO(decoded))  # reopen after verify() (verify closes the file)
print(f"Verified image size: {test_img.size}, mode: {test_img.mode}")

Encoded image length: 69140 characters
Encoded prefix: /9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAgGBgcG...
Verified image size: (512, 512), mode: RGB


## Step 4: Creating the Product Listing Prompt

In [6]:
def create_product_listing_prompt(product_name, price, category, additional_info=None):
    """
    Create a prompt for generating product listings.
    
    Parameters:
    - product_name: Name of the product
    - price: Price of the product
    - category: Product category
    - additional_info: Optional additional information
    
    Returns:
    - Formatted prompt string
    """
    prompt = f"""You are an expert e-commerce copywriter. Analyze the product image and create a compelling product listing.

Product Information:
- Name: {product_name}
- Price: ${price:.2f}
- Category: {category}
{f'- Additional Info: {additional_info}' if additional_info else ''}
 
Please create a professional product listing that includes:
 
1. **Product Title** (catchy, SEO-friendly, 60 characters max)
2. **Product Description** (detailed, 150-200 words)
   - Highlight key features and benefits
   - Use persuasive language
   - Include relevant details visible in the image
3. **Key Features** (bullet points, 5-7 items)
4. **SEO Keywords** (comma-separated, 10-15 relevant keywords)
 
Format your response as JSON with the following structure:
{{
    "title": "Product title here",
    "description": "Full description here",
    "features": ["Feature 1", "Feature 2", ...],
    "keywords": "keyword1, keyword2, ..."
}}
 
Be specific about what you see in the image. Mention colors, materials, design elements, and any distinctive features."""
    
    return prompt

# Test prompt creation — using the real first product's data (row 0), not fabricated values, since we now have real name/price/category.
sample_row = products_df.iloc[0]
sample_price = float(sample_row["Selling Price"].replace("$", "").replace(",", ""))

test_prompt = create_product_listing_prompt(
    product_name=sample_row["Product Name"],
    price=sample_price,
    category=sample_row["Category"],
)
 
print("\n" + "="*50)
print("PROMPT TEMPLATE")
print("="*50)
print(test_prompt[:500] + "...")  # Show first 500 characters


PROMPT TEMPLATE
You are an expert e-commerce copywriter. Analyze the product image and create a compelling product listing.

Product Information:
- Name: Craft-tastic – Empower Poster – Craft Kit – Design a One-of-a-Kind Inspirational Poster
- Price: $14.47
- Category: Toys & Games | Arts & Crafts | Craft Kits | Paper Craft


Please create a professional product listing that includes:

1. **Product Title** (catchy, SEO-friendly, 60 characters max)
2. **Product Description** (detailed, 150-200 words)
   - Highli...


## Step 5: Calling the ChatGPT API with Vision

In [7]:
def generate_product_listing(product_name, price, category, image_url, label="Product"):
    prompt = create_product_listing_prompt(product_name, price, category)
    encoded_image = encode_image_to_base64(image_url)

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0.7,
        max_tokens=1000,
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "text", "text": prompt
                    },
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/jpeg;base64,{encoded_image}",
                            "detail": "high"
                        }
                    }
                ]
            }
        ]
    )

    raw_content = response.choices[0].message.content
    print(f"{label}\n{raw_content}\n")

    cleaned = raw_content.strip().strip("`").replace("json\n", "", 1).strip()

    try:
        listing = json.loads(cleaned)
    except json.JSONDecodeError as e:
        print(f"JSON parsing failed: {e}")
        listing = {"raw_response": raw_content}

    return listing

# Test on the first product
sample_row = products_df.iloc[0]
sample_price = float(sample_row["Selling Price"].replace("$", "").replace(",", ""))

listing = generate_product_listing(
    product_name=sample_row["Product Name"],
    price=sample_price,
    category=sample_row["Category"],
    image_url=sample_row["Image"],
    label="Product listing 1 (row id 0)",
)

print(listing.get("title", "(no title — see raw_response)"))

# Strip the markdown code
print("\n" + "="*50)
print('Converted string into a dictionary')
print("="*50)
print(listing["title"])
print(listing["features"])

Product listing 1 (row id 0)
```json
{
    "title": "Craft-tastic Empower Poster Craft Kit - Unleash Your Creativity",
    "description": "Unleash your child's creativity with the Craft-tastic Empower Poster Craft Kit! This unique craft kit invites kids to design their very own inspirational poster, promoting self-expression and positivity. Measuring 9.75” x 13.75”, the vibrant poster features a kaleidoscope of colors, including bold blues, radiant pinks, and energizing greens, ensuring a stunning visual display. Each kit comes complete with a variety of empowering words and phrases like 'brave', 'happy', and 'unique', encouraging kids to reflect on their own strengths and aspirations. Not only does this kit provide an engaging activity, but it also fosters self-esteem and mindfulness, making it perfect for home, school, or group settings. Let your child's imagination shine as they create a one-of-a-kind poster that inspires others!",
    "features": [
        "Includes vibrant, colorf

**Observations:**

Problem found: When running the model the first time, it said "I'm unable to analyze the image for specific details about what someone is wearing" — meaning it likely didn't actually see the product image. The Hugging Face dataset images are of clothes, accessories, footware, and personal care. It filled in the listing purely from your text metadata (headphones), ignoring the picture entirely.

Solution: I switched from `ashraq/fashion-product-images-small` as the images in the dataset were only 60x80px, too low-res for GPT-4o vision to actually analyze (confirmed via testing). And, switch to `cvnberk/amazon-products` has 500x500 images (fetched via URL) plus real product name/price/category, so we don't have to fabricate metadata.

## Step 6: Processing Multiple Products

In [8]:
#Loop through products and generate listign for each
results = []

for idx, row in products_df.iterrows():
    print(f"Processing product {idx}: {row['Product Name']}")
    try:
        price = float(row["Selling Price"].replace("$", "").replace(",", ""))
        listing = generate_product_listing(
            product_name=row["Product Name"],
            price=price,
            category=row["Category"],
            image_url=row["Image"],
            label=f"Product listing {idx + 1} (row id {idx})",
        )
        results.append({"id": idx, "listing": listing, "error": None})

    except Exception as e:
        print(f"Failed: {e}")
        results.append({"id": idx, "listing": None, "error": str(e)})

Processing product 0: Craft-tastic – Empower Poster – Craft Kit – Design a One-of-a-Kind Inspirational Poster
Product listing 1 (row id 0)
```json
{
    "title": "Craft-tastic Empower Poster Craft Kit - Inspire Creativity",
    "description": "Unleash your child's creativity with the Craft-tastic Empower Poster Craft Kit! This vibrant and engaging kit allows young artists to design a one-of-a-kind inspirational poster that celebrates their uniqueness. Measuring 9.75\" x 13.75\", the poster features a colorful burst of hues, including shades of blue, pink, green, and purple, providing a visually stimulating canvas for self-expression. With an array of empowering words like 'brave', 'unique', and 'happy', children can personalize their poster while boosting their self-esteem. The kit includes all necessary materials, making it easy to create a masterpiece at home or during craft parties. Perfect for ages 6 and up, this craft kit not only promotes artistic skills but also encourages posit

In [9]:
# Save generated listings to a JSON file for submission
with open("generated_listings.json", "w") as f:
    json.dump(results, f, indent=2)

succeeded = sum(1 for r in results if r["error"] is None)
failed = sum(1 for r in results if r["error"] is not None)
print(f"BATCH COMPLETE: {succeeded}/{len(results)} succeeded, {failed} failed")
print("Saved to generated_listings.json")

BATCH COMPLETE: 5/5 succeeded, 0 failed
Saved to generated_listings.json
